In [1]:
# Definimos las rutas en variables de Python
import re

input_dir = "./data/parquet_all"
output_dir = "../data"

# Parámetro principal del timeframe (ejemplos: 10min, 20m, 1h, 4h, 1hora, 4horas)
timeframe = "1h"

# Rango de fechas opcional. Si ambos están vacíos -> toma todo el histórico.
# Ejemplo: start_date = "01/01/2024" y end_date = "02/02/2024"
start_date = "01/01/2015"
end_date = "01/01/2016"

# Lista opcional de divisas.
# Vacío ([]) => todas.
# Ejemplos: ["EURUSD"], ["EURUSD", "GBPUSD", "USDJPY"]
selected_symbols = ['USDJPY']

def parse_timeframe_notebook(raw: str) -> tuple[int, str]:
    token = re.sub(r"\s+", "", raw.strip().lower())
    alias = {
        "m": "m", "min": "m", "mins": "m", "minute": "m", "minutes": "m", "minuto": "m", "minutos": "m",
        "h": "h", "hr": "h", "hrs": "h", "hour": "h", "hours": "h", "hora": "h", "horas": "h",
    }
    m = re.fullmatch(r"(\d+)([a-z]+)", token)
    if not m:
        raise ValueError(f"Timeframe inválido: {raw}. Usa por ejemplo 10min, 20m, 1h o 4h.")
    n = int(m.group(1))
    unit = alias.get(m.group(2))
    if n <= 0 or unit is None:
        raise ValueError(f"Timeframe inválido: {raw}")
    minutes = n if unit == "m" else n * 60
    folder_tag = f"h{minutes // 60}" if minutes % 60 == 0 else f"m{minutes}"
    return minutes, folder_tag

def normalize_symbols_arg(symbols) -> str:
    if symbols is None:
        return ""
    if isinstance(symbols, str):
        return symbols.strip()
    out = [str(s).strip().upper() for s in symbols if str(s).strip()]
    return ",".join(out)

timeframe_minutes, timeframe_folder_tag = parse_timeframe_notebook(timeframe)
parquet_dir = f"{output_dir}/parquet_{timeframe_folder_tag}"
symbols_arg = normalize_symbols_arg(selected_symbols)

print(f"Origen: {input_dir}")
print(f"Destino: {output_dir}")
print(f"Timeframe: {timeframe} -> parquet_{timeframe_folder_tag} ({timeframe_minutes} min)")
if start_date.strip() or end_date.strip():
    print(f"Rango: {start_date or '-inf'} -> {end_date or '+inf'}")
else:
    print("Rango: COMPLETO")
if symbols_arg:
    print(f"Divisas: {symbols_arg}")
else:
    print("Divisas: TODAS")


Origen: ./data/parquet_all
Destino: ../data
Timeframe: 1h -> parquet_h1 (60 min)
Rango: 01/01/2015 -> 01/01/2016
Divisas: USDJPY


In [2]:
# El símbolo "!" le dice a Jupyter que ejecute esto en tu terminal de Linux.
# El símbolo "$" nos permite inyectar las variables de Python.
!python build_h1_dataset.py --input_dir "$input_dir" --output_dir "$output_dir" --timeframe "$timeframe" --start_date "$start_date" --end_date "$end_date" --symbols "$symbols_arg" --fx_only 0 --n_jobs 4


[2026-03-19 17:01:57,549] INFO: Starting build_h1_dataset.py
[2026-03-19 17:01:57,550] INFO: Input dir: ./data/parquet_all
[2026-03-19 17:01:57,550] INFO: Output dir: ../data
[2026-03-19 17:01:57,550] INFO: timeframe=1h (1H, 60 min)
[2026-03-19 17:01:57,550] INFO: date_range_utc=[2015-01-01 00:00:00 .. 2016-01-01 23:59:59.999999]
[2026-03-19 17:01:57,550] INFO: symbols_filter=USDJPY
[2026-03-19 17:01:57,550] INFO: fx_only=False n_jobs=4 force=False no_calendar=False eps_ohlc=1e-07
[2026-03-19 17:01:57,552] INFO: Processing 1 assets...
Building masks: 100%|█████████████████████████████| 1/1 [00:00<00:00, 29.46it/s]
[2026-03-19 17:01:58,364] INFO: Calendar+Masks written. total_periods_in_range=8740
[2026-03-19 17:01:58,364] INFO: OK: built 1H dataset in ../data
[2026-03-19 17:01:58,365] INFO: - assets: 1
[2026-03-19 17:01:58,365] INFO: - manifest: ../data/manifest.json
[2026-03-19 17:01:58,365] INFO: - report: ../data/data_quality_report.md


In [3]:
!python integrity_checks.py --dataset_dir "$output_dir" --timeframe "$timeframe" --symbols "$symbols_arg"


Filtro de símbolos: USDJPY
Iniciando auditoría de 1 archivos...

Check: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 75.86file/s]

RESUMEN DE INTEGRIDAD
✅ TODOS LOS ARCHIVOS PASARON LA PRUEBA.


In [4]:
!python integrity_checks.py --dataset_dir "$output_dir" --timeframe "$timeframe"


Filtro de símbolos: ALL
Iniciando auditoría de 1 archivos...

Check: 100%|████████████████████████████████████| 1/1 [00:00<00:00, 80.89file/s]

RESUMEN DE INTEGRIDAD
✅ TODOS LOS ARCHIVOS PASARON LA PRUEBA.


In [7]:
import pandas as pd

# Carga el archivo problemático
ruta_eurusd = f"{parquet_dir}/USDJPY.parquet"
df = pd.read_parquet(ruta_eurusd)

# Filtra las filas donde cualquier precio sea <= 0
bad_rows = df[(df['open'] <= 0) | (df['high'] <= 0) | (df['low'] <= 0) | (df['close'] <= 0)]

print(f"Se encontraron {len(bad_rows)} velas corruptas.")
display(bad_rows)


Se encontraron 0 velas corruptas.


,timestamp,open,high,low,close,volume,n_m1,delta_vol,close_strength,vwap,vwap_distance,efficiency_ratio,volatility,is_complete,gap_bars,gap_hours,is_gap,is_weekend_gap,is_data_gap


In [5]:
import pandas as pd

# 1. Definir la ruta exacta del archivo problemático
ruta_eurusd = f"{parquet_dir}/EURUSD.parquet"

# 2. Cargar los datos
df = pd.read_parquet(ruta_eurusd)
filas_antes = len(df)

# 3. Identificar y filtrar los precios imposibles
valid_prices = (df['open'] > 0) & (df['high'] > 0) & (df['low'] > 0) & (df['close'] > 0)
df_clean = df[valid_prices].copy()

# 4. RECALCULAR LOS GAPS
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])
dt_minutes = df_clean["timestamp"].diff().dt.total_seconds().div(60.0)
df_clean["gap_bars"] = (dt_minutes / timeframe_minutes).round().fillna(1).astype(int)
df_clean["gap_hours"] = dt_minutes.div(60.0).fillna(timeframe_minutes / 60.0)
df_clean["is_gap"] = df_clean["gap_bars"] > 1

# 5. Sobrescribir el archivo ya curado
df_clean.to_parquet(ruta_eurusd, compression="zstd", index=False)

filas_despues = len(df_clean)
print("✅ Cirugía completada.")
print(f"   - Filas originales: {filas_antes}")
print(f"   - Filas eliminadas: {filas_antes - filas_despues}")
print("   - El archivo EURUSD.parquet ha sido sobrescrito y curado.")


✅ Cirugía completada.
   - Filas originales: 155390
   - Filas eliminadas: 1
   - El archivo EURUSD.parquet ha sido sobrescrito y curado.


In [6]:
ruta = f"{parquet_dir}/XAUUSD.parquet"
df = pd.read_parquet(ruta)
df.head()


,timestamp,open,high,low,close,volume,n_m1,delta_vol,close_strength,vwap,vwap_distance,efficiency_ratio,volatility,is_complete,gap_hours,is_gap,is_weekend_gap,is_data_gap
0,2009-03-15 17:00:00,929.599976,929.599976,929.599976,929.599976,0.0,1,0.0,0.500000,929.599976,0.0,0.000000,0.000000,False,1,False,False,False
1,2009-03-15 18:00:00,925.849976,928.099976,925.349976,925.900024,0.0,58,0.0,0.200018,925.900024,0.0,0.003208,0.525427,False,1,False,False,False
2,2009-03-15 19:00:00,926.099976,927.049988,924.549988,926.349976,0.0,56,0.0,0.719995,926.349976,0.0,0.034014,0.397697,False,1,False,False,False
3,2009-03-15 20:00:00,926.349976,928.400024,923.349976,928.330017,0.0,60,0.0,0.986137,928.330017,0.0,0.138272,0.917578,True,1,False,False,False
4,2009-03-15 21:00:00,928.400024,928.799988,926.030029,926.349976,0.0,59,0.0,0.115506,926.349976,0.0,0.239495,0.197534,False,1,False,False,False


In [7]:
df.tail()

,timestamp,open,high,low,close,volume,n_m1,delta_vol,close_strength,vwap,vwap_distance,efficiency_ratio,volatility,is_complete,gap_hours,is_gap,is_weekend_gap,is_data_gap
99552,2025-12-05 06:00:00,4222.757812,4225.555176,4219.834961,4223.524902,0.0,60,0.0,0.645070,4223.524902,0.0,0.025886,0.746498,True,1,False,False,False
99553,2025-12-05 07:00:00,4223.524902,4229.584961,4222.634766,4228.544922,0.0,60,0.0,0.850358,4228.544922,0.0,0.132598,1.010337,True,1,False,False,False
99554,2025-12-05 08:00:00,4228.575195,4238.705078,4221.285156,4234.365234,0.0,60,0.0,0.750869,4234.365234,0.0,0.081005,4.345162,True,1,False,False,False
99555,2025-12-05 09:00:00,4234.345215,4245.198242,4228.044922,4241.495117,0.0,60,0.0,0.784116,4241.495117,0.0,0.113654,4.168202,True,1,False,False,False
99556,2025-12-05 10:00:00,4241.484863,4259.104980,4233.944824,4241.535156,0.0,59,0.0,0.301681,4241.535156,0.0,0.000712,6.073507,False,1,False,False,False


In [8]:
df.describe()

,timestamp,open,high,low,close,volume,n_m1,delta_vol,close_strength,vwap,vwap_distance,efficiency_ratio,volatility,gap_hours
count,99557,99557.000000,99557.000000,99557.000000,99557.000000,99557.0,99557.000000,99557.0,99557.000000,99557.000000,99557.0,99557.000000,99557.000000,99557.000000
mean,2017-06-14 12:27:00.952821248,1614.838863,1616.983748,1612.648378,1614.866941,0.0,58.833472,0.0,0.507875,1614.866941,0.0,0.128563,0.975386,1.472624
min,2009-03-15 17:00:00,861.000000,867.729980,861.000000,866.349976,0.0,1.000000,0.0,0.000000,866.349976,0.0,0.000000,0.000000,1.000000
25%,2013-04-02 02:00:00,1247.849976,1249.209961,1246.439941,1247.859985,0.0,60.000000,0.0,0.258826,1247.859985,0.0,0.050637,0.415613,1.000000
50%,2017-05-30 21:00:00,1429.550049,1431.194946,1427.604980,1429.628052,0.0,60.000000,0.0,0.509045,1429.628052,0.0,0.107699,0.683700,1.000000
75%,2021-08-09 02:00:00,1806.425049,1808.449951,1804.498047,1806.427979,0.0,60.000000,0.0,0.760322,1806.427979,0.0,0.183429,1.176097,1.000000
max,2025-12-05 10:00:00,4377.884766,4381.297852,4358.948242,4377.848145,0.0,60.000000,0.0,1.000000,4377.848145,0.0,5.987805,26.526881,77.000000
std,NaN,562.707670,563.721565,561.673988,562.770233,0.0,6.300442,0.0,0.288952,562.770233,0.0,0.105605,1.083903,4.684530


In [ ]:
import pandas as pd

# 1. Cargar la máscara
ruta_mask = "../data/availability_masks.parquet"
masks = pd.read_parquet(ruta_mask)

# 2. Convertir el string a datetime (formato naive, tal como lo guarda el script)
fecha_glitch = pd.to_datetime("2001-09-11 20:00:00")

# 3. Buscar esa hora exacta y apagar (False) el valor de EURUSD
filtro_fecha = masks['timestamp'] == fecha_glitch#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import logging
from multiprocessing import Pool
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
import yaml
import pandas as pd

from token_features import compute_asset_tokens
from masking import load_calendar_and_presence, build_liveness_masks, write_parquet
from global_features import compute_global_tokens, write_parquet as write_parquet_global
from build_universe_spec import parse_fx_symbol 

# Configuración de Logging Institucional
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

def _read_manifest(dataset_dir: Path) -> dict:
    m = dataset_dir / "manifest.json"
    if not m.exists():
        raise SystemExit(f"Error: manifest.json no encontrado en {dataset_dir}")
    return json.loads(m.read_text(encoding="utf-8"))

def _load_specs(universe_spec: Path, token_spec: Path):
    uni = json.loads(universe_spec.read_text(encoding="utf-8"))
    tok = yaml.safe_load(token_spec.read_text(encoding="utf-8"))
    return uni, tok

def _usd_pairs_from_canonical(canonical: list[str]):
    """Identifica pares USD usando el parser oficial del universo."""
    pairs = []
    for s in canonical:
        base, quote = parse_fx_symbol(s)
        if not base:
            continue
        if base == "USD":
            pairs.append((s.lower(), +1)) 
        elif quote == "USD":
            pairs.append((s.lower(), -1)) 
    return pairs

def _build_one(task):
    """Worker resiliente para procesamiento paralelo con optimización float32."""
    sym, in_path, out_path = task
    try:
        # Lectura de datos crudos de la Fase 1
        df = pq.read_table(str(in_path)).to_pandas()
        
        # Generación de tokens técnicos (Fase 2)
        tokens = compute_asset_tokens(df)
        
        # Optimización de Memoria: float32 para eficiencia en entrenamiento
        float_cols = tokens.select_dtypes(include=['float64']).columns
        tokens[float_cols] = tokens[float_cols].astype('float32')
        
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table = pa.Table.from_pandas(tokens, preserve_index=False)
        pq.write_table(table, str(out_path), compression="zstd")
        
        return sym, len(tokens), True
    except Exception as e:
        logger.error(f"Fallo crítico en activo {sym}: {str(e)}")
        return sym, str(e), False

def main():
    ap = argparse.ArgumentParser(description="Orquestador de Fase 2: Tokens y Máscaras")
    ap.add_argument("--dataset_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--universe_spec", required=True)
    ap.add_argument("--token_spec", required=True)
    ap.add_argument("--n_jobs", default=4, type=int)
    ap.add_argument("--global_missing_thr", default=0.30, type=float)
    args = ap.parse_args()

    dataset_dir = Path(args.dataset_dir)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1. Carga de metadatos y normalización de nombres
    uni, _ = _load_specs(Path(args.universe_spec), Path(args.token_spec))
    canonical = [s.lower() for s in uni["canonical_order"]]
    usd_pairs = _usd_pairs_from_canonical(canonical)
    
    # Carga de archivos maestros con normalización de columnas (Case-Insensitive)
    calendar, presence = load_calendar_and_presence(dataset_dir)
    manifest = _read_manifest(dataset_dir)

    # 2. Construcción de Máscaras Causalidad/Liveness
    liveness = build_liveness_masks(calendar, manifest, canonical)
    attention = liveness.copy()

    # Preservar especificaciones para trazabilidad
    spec_dir = out_dir / "spec"
    spec_dir.mkdir(parents=True, exist_ok=True)
    (spec_dir / "UniverseSpec_v1.json").write_text(json.dumps(uni, indent=2))
    (spec_dir / "TokenSpec_v2.yaml").write_text(Path(args.token_spec).read_text())

    # 3. Exportar Archivos Maestros de Máscaras
    write_parquet(calendar, out_dir / "calendar_master.parquet")
    write_parquet(presence, out_dir / "availability_masks.parquet")
    write_parquet(liveness, out_dir / "liveness_masks.parquet")
    write_parquet(attention, out_dir / "attention_masks.parquet")

    # 4. Procesamiento Paralelo de Asset Tokens
    pdir = dataset_dir / "parquet_h1"
    asset_out_dir = out_dir / "asset_tokens"
    asset_out_dir.mkdir(parents=True, exist_ok=True)

    # SOLUCIÓN CRÍTICA: Mapeo de archivos en disco insensible a mayúsculas
    files_on_disk = {p.stem.lower(): p for p in pdir.glob("*.parquet")}

    tasks = []
    asset_token_paths = {}
    for sym in canonical:
        sym_low = sym.lower()
        if sym_low in files_on_disk:
            in_path = files_on_disk[sym_low]
            out_path = asset_out_dir / f"{sym_low}.parquet"
            tasks.append((sym_low, in_path, out_path))
            asset_token_paths[sym_low] = out_path
        else:
            logger.warning(f"Activo {sym_low} definido en el universo pero no encontrado en {pdir}")

    if not tasks:
        raise SystemExit(f"Error: No se encontraron archivos para procesar en {pdir}. Verifica nombres.")

    logger.info(f"Procesando {len(tasks)} activos con {args.n_jobs} hilos paralelos...")

    processed_count = 0
    with Pool(processes=args.n_jobs) as pool:
        for sym, result, success in pool.imap_unordered(_build_one, tasks):
            if success:
                processed_count += 1
                if processed_count % 10 == 0:
                    logger.info(f"Progreso: {processed_count}/{len(tasks)} activos procesados.")
            else:
                logger.error(f"Error en {sym}: {result}")

    # 5. Generación de Global Tokens (Market Sentiment)
    logger.info("Calculando Global Tokens (Market Mood)...")
    global_tokens = compute_global_tokens(
        calendar=calendar,
        presence_masks=presence,
        canonical_order=canonical,
        asset_token_paths=asset_token_paths,
        usd_pairs=usd_pairs,
        global_missing_thr=args.global_missing_thr,
    )
    
    # Conversión a float32 para coherencia de memoria
    g_float_cols = global_tokens.select_dtypes(include=['float64']).columns
    global_tokens[g_float_cols] = global_tokens[g_float_cols].astype('float32')
    
    write_parquet_global(global_tokens, out_dir / "global_tokens.parquet")

    # 6. Resumen Final
    summary = {
        "assets_processed": processed_count,
        "total_rows_global": len(global_tokens),
        "status": "Success" if processed_count > 0 else "No assets processed"
    }
    (out_dir / "phase2_build_summary.json").write_text(json.dumps(summary, indent=2))
    logger.info(f"✅ Proceso completado. Resumen: {summary}")

if __name__ == "__main__":
    main()

# Verificamos si la encontró
if filtro_fecha.sum() > 0:
    masks.loc[filtro_fecha, 'EURUSD'] = False
    # Guardar de nuevo
    masks.to_parquet(ruta_mask, compression="zstd", index=False)
    print("✅ availability_masks.parquet curado. El hueco del 9/11 ahora está marcado como False.")
else:
    print("⚠️ No se encontró la fecha en la máscara. Revisa el formato.")

✅ availability_masks.parquet curado. El hueco del 9/11 ahora está marcado como False.
